<a href="https://colab.research.google.com/github/Vishrutha333/AI_LAB/blob/main/1BM24CS333_Week6_PropositionalLogic_TruthTableEnumeration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:

def evaluate_sentence(sentence, model):

    if isinstance(sentence, str):
        return model[sentence]

    operator = sentence[0]

    if operator == 'NOT':
        return not evaluate_sentence(sentence[1], model)

    elif operator == 'AND':
        return (evaluate_sentence(sentence[1], model) and
                evaluate_sentence(sentence[2], model))

    elif operator == 'OR':
        return (evaluate_sentence(sentence[1], model) or
                evaluate_sentence(sentence[2], model))

    elif operator == 'IMPLIES':
        return (not evaluate_sentence(sentence[1], model) or
                evaluate_sentence(sentence[2], model))


def extract_symbols(sentence):

    if isinstance(sentence, str):
        return {sentence}

    symbols = set()

    for part in sentence[1:]:
        if isinstance(part, tuple):
            symbols.update(extract_symbols(part))
        elif isinstance(part, str):
            symbols.add(part)

    return symbols


def extend_model(model, symbol, value):

    new_model = model.copy()
    new_model[symbol] = value
    return new_model


def TT_Check_All(KB, alpha, symbols, model):

    if len(symbols) == 0:

        if evaluate_sentence(KB, model):
            return evaluate_sentence(alpha, model)
        else:
            return True

    p = symbols[0]
    rest = symbols[1:]

    model_true = extend_model(model, p, True)
    eval_true = TT_Check_All(KB, alpha, rest, model_true)

    model_false = extend_model(model, p, False)
    eval_false = TT_Check_All(KB, alpha, rest, model_false)

    return eval_true and eval_false


def TT_Entails(KB, alpha):

    symbols = sorted(extract_symbols(KB) | extract_symbols(alpha))

    return TT_Check_All(KB, alpha, symbols, {})


def tokenize(expression):

    expression = expression.replace(" ", "")
    tokens = []
    i = 0

    while i < len(expression):

        if expression.startswith("->", i):
            tokens.append("->")
            i += 2

        elif expression[i] in "^v-()":
            tokens.append(expression[i])
            i += 1

        elif expression[i].isupper():
            tokens.append(expression[i])
            i += 1

        else:
            raise ValueError("Invalid character: " + expression[i])

    return tokens


def parse_expression(tokens):

    pos = 0

    def parse_atom():
        nonlocal pos

        if pos >= len(tokens):
            raise ValueError("Unexpected end of expression")

        token = tokens[pos]

        if token == '-':
            pos += 1
            return ('NOT', parse_atom())

        elif token == '(':
            pos += 1
            result = parse_implies()

            if pos >= len(tokens) or tokens[pos] != ')':
                raise ValueError("Missing closing bracket")

            pos += 1
            return result

        elif token not in ('^', 'v', '->', ')'):
            pos += 1
            return token

        raise ValueError("Unexpected token: " + token)

    def parse_and():
        nonlocal pos

        left = parse_atom()

        while pos < len(tokens) and tokens[pos] == '^':
            pos += 1
            right = parse_atom()
            left = ('AND', left, right)

        return left

    def parse_or():
        nonlocal pos

        left = parse_and()

        while pos < len(tokens) and tokens[pos] == 'v':
            pos += 1
            right = parse_and()
            left = ('OR', left, right)

        return left

    def parse_implies():
        nonlocal pos

        left = parse_or()

        if pos < len(tokens) and tokens[pos] == '->':
            pos += 1
            right = parse_implies()
            left = ('IMPLIES', left, right)

        return left

    result = parse_implies()

    if pos != len(tokens):
        raise ValueError("Unexpected token: " + tokens[pos])

    return result


def display_truth_table(KB, alpha):

    symbols = sorted(extract_symbols(KB) | extract_symbols(alpha))

    print("\nTRUTH TABLE")
    print("-" * 65)

    for symbol in symbols:
        print(f"{symbol:^8}", end="")

    print(f"{'KB':^10}{'Alpha':^10}{'KB => Alpha':^15}")
    print("-" * 65)

    total = 2 ** len(symbols)
    all_entail = True

    for i in range(total):

        model = {}

        for j, symbol in enumerate(symbols):
            model[symbol] = not bool(
                (i >> (len(symbols) - j - 1)) & 1
            )

        kb_value = evaluate_sentence(KB, model)
        alpha_value = evaluate_sentence(alpha, model)

        entails_value = (not kb_value) or alpha_value

        if not entails_value:
            all_entail = False

        for symbol in symbols:
            print(f"{str(model[symbol]):^8}", end="")

        print(f"{str(kb_value):^10}"
              f"{str(alpha_value):^10}"
              f"{str(entails_value):^15}")

    print("-" * 65)

    return all_entail


try:

    print("PROPOSITIONAL LOGIC: ")

    KB_input = input("Enter Knowledge Base (KB): ")
    alpha_input = input("Enter Alpha: ")

    KB_tokens = tokenize(KB_input)
    alpha_tokens = tokenize(alpha_input)

    KB = parse_expression(KB_tokens)
    alpha = parse_expression(alpha_tokens)

    result = display_truth_table(KB, alpha)

    print("\nRESULT")
    print("--------------------------------")

    if result:
        print("KB entails Alpha: TRUE")
    else:
        print("KB entails Alpha: FALSE")

except (ValueError, IndexError) as e:
    print("Invalid logical expression:", e)


PROPOSITIONAL LOGIC - TT ENTAILS
--------------------------------
Enter Knowledge Base (KB): ((A v C) ^ (B v -C))
Enter Alpha: A v B

TRUTH TABLE
-----------------------------------------------------------------
   A       B       C        KB      Alpha     KB => Alpha  
-----------------------------------------------------------------
  True    True    True     True      True        True      
  True    True   False     True      True        True      
  True   False    True    False      True        True      
  True   False   False     True      True        True      
 False    True    True     True      True        True      
 False    True   False    False      True        True      
 False   False    True    False     False        True      
 False   False   False    False     False        True      
-----------------------------------------------------------------

RESULT
--------------------------------
KB entails Alpha: TRUE
